# PhobosStream + MiniMax-H3 (5.88s 24fps Video + 32 kHz Stereo Audio on Free T4)

Run **MiniMax-H3** (`32B Qwen3-VL` + `20B Pruned W6A8` 4-Step Turbo + `4.85 GB` 36-block `ViT3D` Video VAE + `32 kHz` Stereo BigVGAN Audio VAE) on a **Free Google Colab Tesla T4 (`14.56 GiB` VRAM + `12 GiB` RAM)** without Out-Of-Memory crashes.

**Verified Benchmarks (Free Colab T4 / 12 GiB RAM):**
- **36-Block `4.85 GB` `ViT3D` Video VAE Decode (`141` frames):** Stock T4 crashes (`>14.56 GiB OOM`) -> PhobosStream **`466.8 MB` Peak VRAM** (`96.8%` saved)
- **Stereo `32 kHz` BigVGAN Audio VAE:** Stock T4 crashes (`>14.56 GiB OOM`) -> PhobosStream chunked overlap-add **`618.4 MB` Peak VRAM** (`8.15 s`)
- **Honest Timing Note:** Full end-to-end 5.88s generation takes **~25 min** on a free 2018 T4 + `12 GiB` RAM (mostly Stage 2 Turing compute + RAM ceiling); faster on L4/A100 or GPUs with >=32 GB host RAM.

GitHub: [martianacademy/phobos-stream](https://github.com/martianacademy/phobos-stream) | ComfyUI Node: [martianacademy/ComfyUI-PhobosStream](https://github.com/martianacademy/ComfyUI-PhobosStream)

In [ ]:
# 1. Verify GPU, PyTorch CUDA build, and free disk/RAM
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, shutil, psutil
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('Host RAM (GiB):', round(psutil.virtual_memory().total / 1e9, 2))
print('Free Disk (GiB):', round(shutil.disk_usage('/content').free / 1e9, 2))

In [ ]:
# 2. Install PhobosStream & ComfyUI (preserving Colab's native torch+cu130 build)
!pip install -q --upgrade phobos-stream huggingface_hub
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!grep -vE '^(torch|torchvision|torchaudio)\b' /content/ComfyUI/requirements.txt > /tmp/comfy_reqs.txt
!pip install -q -r /tmp/comfy_reqs.txt
import phobos_stream
print('PhobosStream version:', phobos_stream.__version__)

In [ ]:
# 3. Download ungated MiniMax-H3 open weights from Comfy-Org/MiniMax-H3 (~34 GB total)
from pathlib import Path
from huggingface_hub import hf_hub_download

MODELS = Path('/content/ComfyUI/models')
DOWNLOADS = [
    ('split_files/text_encoders/qwen_3_vl_32b_nvfp4.safetensors', MODELS / 'text_encoders'),
    ('split_files/diffusion_models/minimax_h3_pruned_w6a8_cu130.safetensors', MODELS / 'diffusion_models'),
    ('split_files/loras/minimax_h3_turbo_4steps.safetensors', MODELS / 'loras'),
    ('split_files/vae/minimax_h3_video_vae_bf16.safetensors', MODELS / 'vae'),
    ('split_files/vae/minimax_h3_audio_vae_bf16.safetensors', MODELS / 'vae'),
]
for repo_path, dst_dir in DOWNLOADS:
    dst_dir.mkdir(parents=True, exist_ok=True)
    print('Downloading:', repo_path)
    hf_hub_download(repo_id='Comfy-Org/MiniMax-H3', filename=repo_path, local_dir='/content/hf_h3')
    src = Path('/content/hf_h3') / repo_path
    dst = dst_dir / src.name
    if not dst.exists():
        dst.symlink_to(src)
print('All MiniMax-H3 weights ready!')

In [ ]:
# 4. Configure your prompt, resolution & streaming mode
PROMPT = (
    'Cinematic IMAX tracking shot on Mars at golden hour: an astronaut in a detailed white '
    'spacesuit stands beside twin dust-covered rovers as a silver spacecraft descends with '
    'glowing blue thrusters. Ambient wind and deep thruster rumble.'
)
WIDTH = 736
HEIGHT = 416
NUM_FRAMES = 141  # 5.88s @ 24fps (use 49 or 73 for faster ~2-3s test runs)
SEED = 42
# 'hybrid_budget' keeps as many ViT3D VAE blocks GPU-resident as fit for faster decode;
# switch to 'cpu_offload' for strict <500 MB VAE Peak VRAM.
VAE_STREAM_MODE = 'hybrid_budget'
print(f'Configured: {WIDTH}x{HEIGHT}, {NUM_FRAMES} frames @ 24fps, VAE mode={VAE_STREAM_MODE}')

In [ ]:
# 5. Run isolated 3-stage pipeline (Stage 1: Qwen3-VL -> Stage 2: MiniMax-H3 4-Step -> Stage 3A: PhobosStream ViT3D VAE -> Stage 3B: Chunked Stereo Audio)
import os, subprocess, sys
from pathlib import Path

runner = f'''
import gc, os, sys, time, subprocess
from pathlib import Path
sys.path.insert(0, '/content/ComfyUI')
import torch
from phobos_stream.integrations.comfyui.vae_patcher import attach_streamer_to_comfy_vae

def _unwrap(out):
    return out.result if hasattr(out, 'result') else out

stage = sys.argv[1]
work = Path('/content/phobos_h3_work')
work.mkdir(exist_ok=True)

if stage == 'stage1':
    import nodes
    t0 = time.perf_counter()
    clip, = nodes.CLIPLoader().load_clip('qwen_3_vl_32b_nvfp4.safetensors', 'minimax_h3')
    pos, = nodes.CLIPTextEncode().encode(clip, {PROMPT!r})
    neg, = nodes.ConditioningZeroOut().zero_out(pos)
    torch.save({{'pos': pos, 'neg': neg}}, work / 'cond.pt')
    print(f'[Stage 1] Done in {{time.perf_counter() - t0:.1f}}s | Peak VRAM: {{torch.cuda.max_memory_allocated()/1e6:.1f}} MB')

elif stage == 'stage2':
    sys.argv = ['main.py', '--lowvram', '--reserve-vram', '6.0']
    import nodes
    from comfy_extras import nodes_minimax
    t0 = time.perf_counter()
    cond = torch.load(work / 'cond.pt', weights_only=False)
    model, = nodes.UNETLoader().load_unet('minimax_h3_pruned_w6a8_cu130.safetensors', 'default')
    model, = nodes.LoraLoaderModelOnly().load_lora_model_only(model, 'minimax_h3_turbo_4steps.safetensors', 1.0)
    latent = _unwrap(nodes_minimax.EmptyMiniMaxH3LatentAV.execute({WIDTH}, {HEIGHT}, {NUM_FRAMES}, 24, 1))[0]
    sampled, = nodes.KSampler().sample(model, {SEED}, 4, 1.0, 'euler', 'simple', cond['pos'], cond['neg'], latent, denoise=1.0)
    torch.save(sampled, work / 'sampled.pt')
    print(f'[Stage 2] Done in {{time.perf_counter() - t0:.1f}}s | Peak VRAM: {{torch.cuda.max_memory_allocated()/1e6:.1f}} MB')

elif stage == 'stage3a':
    import nodes
    from comfy_extras import nodes_audio
    t0 = time.perf_counter()
    sampled = torch.load(work / 'sampled.pt', weights_only=False)
    vid_lat, aud_lat = _unwrap(nodes_audio.SeparateAVLatent.execute(sampled))
    torch.save(aud_lat, work / 'aud_lat.pt')
    vae, = nodes.VAELoader().load_vae('minimax_h3_video_vae_bf16.safetensors')
    meta = attach_streamer_to_comfy_vae(vae, block_stream_mode={VAE_STREAM_MODE!r}, drop_encoder=True)
    print('[Stage 3A] PhobosStream VAE plan:', meta['tier_summary'])
    frames = vae.decode(vid_lat['samples'])
    if frames.ndim == 5: frames = frames[0]
    frames_u8 = (frames.clamp(0, 1).mul(255).to(torch.uint8).cpu().numpy())
    torch.save(frames_u8, work / 'frames_u8.pt')
    print(f'[Stage 3A] Done in {{time.perf_counter() - t0:.1f}}s | Peak VRAM: {{torch.cuda.max_memory_allocated()/1e6:.1f}} MB')

elif stage == 'stage3b':
    import wave, numpy as np, nodes
    t0 = time.perf_counter()
    aud_lat = torch.load(work / 'aud_lat.pt', weights_only=False)
    avae, = nodes.VAELoader().load_vae('minimax_h3_audio_vae_bf16.safetensors')
    attach_streamer_to_comfy_vae(avae, block_stream_mode='stage_only', drop_encoder=True, audio_chunk_steps=64, audio_overlap_steps=8)
    wav = avae.decode(aud_lat['samples'])[0].clamp(-1.0, 1.0).numpy()
    wav_i16 = (wav.T * 32767.0).astype(np.int16)
    with wave.open(str(work / 'audio.wav'), 'wb') as wf:
        wf.setnchannels(wav_i16.shape[1]); wf.setsampwidth(2); wf.setframerate(32000); wf.writeframes(wav_i16.tobytes())
    frames_u8 = torch.load(work / 'frames_u8.pt', weights_only=False)
    h, w = frames_u8.shape[1], frames_u8.shape[2]
    proc = subprocess.Popen(['ffmpeg', '-y', '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', f'{{w}}x{{h}}', '-r', '24', '-i', '-', '-i', str(work / 'audio.wav'), '-c:v', 'libx264', '-crf', '18', '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '192k', '-shortest', '/content/minimax_h3_phobos_output.mp4'], stdin=subprocess.PIPE)
    proc.stdin.write(frames_u8.tobytes()); proc.stdin.close(); proc.wait()
    print(f'[Stage 3B] Muxed /content/minimax_h3_phobos_output.mp4 in {{time.perf_counter() - t0:.1f}}s | Peak VRAM: {{torch.cuda.max_memory_allocated()/1e6:.1f}} MB')
'''
Path('/content/run_h3_stage.py').write_text(runner)

for st, extra_env in [
    ('stage1', {}),
    ('stage2', {'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}),
    ('stage3a', {}),
    ('stage3b', {}),
]:
    env = dict(os.environ, **extra_env)
    print(f'=== Running {st} ===')
    subprocess.run([sys.executable, '/content/run_h3_stage.py', st], env=env, check=True)

In [ ]:
# 6. Play the generated video + 32 kHz stereo audio inline!
from IPython.display import HTML
from base64 import b64encode

mp4_bytes = open('/content/minimax_h3_phobos_output.mp4', 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
HTML(f'<video width=640 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>')